# 11 — Doubly Robust (T9)


In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

import warnings
warnings.filterwarnings("ignore")

import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

sys.path.insert(0, str(Path("..").resolve()))
from src.dataset import OpenBanditDataset
from src.propensity import OBD_COMMON_CONTEXT_DIM, train_propensity_model, extract_propensity_scores
from src.estimators import ips_with_clipping, snips_estimate
from src.evaluation import bootstrap_estimates, bias_variance_mse

from obp.ope import (
    DirectMethod,
    InverseProbabilityWeighting,
    SelfNormalizedInverseProbabilityWeighting,
    DoublyRobust,
    OffPolicyEvaluation,
)

sns.set_theme(style="whitegrid")
np.random.seed(42)

N_ACTIONS    = 80
N_FEATURES   = OBD_COMMON_CONTEXT_DIM  # 20
LEN_LIST     = 3
RANDOM_STATE = 42
N_BOOTSTRAP  = 200
PI_EVAL      = 1.0 / N_ACTIONS
V_STAR       = 0.0038  # ground truth proxy

FIGURES_DIR = Path("../figures/week9")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


## Dane i modele (baseline)


In [ ]:
ds_random = OpenBanditDataset(behavior_policy="random", campaign="all")
fb_random = ds_random.obtain_batch_bandit_feedback()
ds_bts = OpenBanditDataset(behavior_policy="bts", campaign="all")
fb_bts = ds_bts.obtain_batch_bandit_feedback()

ctx_random = fb_random["context"][:, :N_FEATURES]
act_random = fb_random["action"]
ctx_bts    = fb_bts["context"][:, :N_FEATURES]
act_bts    = fb_bts["action"]
rew_bts    = fb_bts["reward"].astype(np.float64)
pos_bts    = fb_bts["position"]

pscores_bts = np.load("../results/week5/pscores_bts.npy")

X_full = np.hstack([ctx_random, np.eye(N_ACTIONS)[act_random]])
X_tr, X_val, y_tr, y_val = train_test_split(
    X_full, fb_random["reward"].astype(np.float64),
    test_size=0.2, random_state=RANDOM_STATE,
    stratify=fb_random["reward"]
)
reward_model = XGBClassifier(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric="logloss", early_stopping_rounds=20,
    tree_method="hist", n_jobs=1, random_state=RANDOM_STATE,
)
reward_model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
print(f"Reward model OK. best_iter={reward_model.best_iteration}")

n_rounds = len(rew_bts)
er_matrix = np.zeros((n_rounds, N_ACTIONS, LEN_LIST), dtype=np.float64)
for a in range(N_ACTIONS):
    X_a = np.hstack([ctx_bts, np.eye(N_ACTIONS)[[a] * n_rounds]])
    er_matrix[:, a, :] = reward_model.predict_proba(X_a)[:, 1:2]  # broadcast do LEN_LIST

action_dist_eval = np.full((n_rounds, N_ACTIONS, LEN_LIST), PI_EVAL, dtype=np.float64)

print(f"er_matrix shape: {er_matrix.shape}")
print(f"pscores: min={pscores_bts.min():.4f}  max={pscores_bts.max():.4f}")


## DR — baseline (dobry PS + dobry reward model)


In [ ]:
fb_bts_ope = {**fb_bts, "pscore": pscores_bts.astype(np.float64)}

ope = OffPolicyEvaluation(
    bandit_feedback=fb_bts_ope,
    ope_estimators=[
        DirectMethod(),
        InverseProbabilityWeighting(),
        SelfNormalizedInverseProbabilityWeighting(),
        DoublyRobust(),
    ],
)

policy_values = ope.estimate_policy_values(
    action_dist=action_dist_eval,
    estimated_rewards_by_reg_model=er_matrix,
)

ci_results = ope.estimate_intervals(
    action_dist=action_dist_eval,
    estimated_rewards_by_reg_model=er_matrix,
    n_bootstrap_samples=N_BOOTSTRAP,
    random_state=RANDOM_STATE,
)

print(f"{'Estimator':8s}  {'V̂':10s}  {'CI Lower':10s}  {'CI Upper':10s}")
for name in ["dm", "ipw", "snipw", "dr"]:
    v  = policy_values[name]
    ci = ci_results[name]
    print(f"{name.upper():8s}  {v:.6f}  {ci['95.0% CI (lower)']:.6f}  {ci['95.0% CI (upper)']:.6f}")


## Eksperyment A: zły PS


In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
pscores_bad = rng.uniform(0.001, 0.05, size=len(pscores_bts))

fb_bad_ps = {**fb_bts, "pscore": pscores_bad.astype(np.float64)}
ope_bad_ps = OffPolicyEvaluation(
    bandit_feedback=fb_bad_ps,
    ope_estimators=[
        DirectMethod(),
        InverseProbabilityWeighting(),
        DoublyRobust(),
    ],
)

vals_bad_ps = ope_bad_ps.estimate_policy_values(
    action_dist=action_dist_eval,
    estimated_rewards_by_reg_model=er_matrix,
)

print(f"{'Estimator':8s}  {'V̂ (baseline)':14s}  {'V̂ (bad PS)':12s}  {'Zmiana':10s}")
for name in ["dm", "ipw", "dr"]:
    v_base = policy_values[name]
    v_bad  = vals_bad_ps[name]
    print(f"{name.upper():8s}  {v_base:.6f}         {v_bad:.6f}      {v_bad - v_base:+.6f}")
print(f"  DR(bad PS)  = {vals_bad_ps['dr']:.6f}")
print(f"  DM(good)    = {policy_values['dm']:.6f}")
print(f"  V*          = {V_STAR}")


## Eksperyment B: Zły reward model → czy IPS ratuje DR?


In [ ]:
er_bad = np.full_like(er_matrix, fill_value=0.5)
fb_good_ps = {**fb_bts, "pscore": pscores_bts.astype(np.float64)}
ope_bad_dm = OffPolicyEvaluation(
    bandit_feedback=fb_good_ps,
    ope_estimators=[
        DirectMethod(),
        InverseProbabilityWeighting(),
        DoublyRobust(),
    ],
)

vals_bad_dm = ope_bad_dm.estimate_policy_values(
    action_dist=action_dist_eval,
    estimated_rewards_by_reg_model=er_bad,
)

print(f"{'Estimator':8s}  {'V̂ (baseline)':14s}  {'V̂ (bad DM)':12s}  {'Zmiana':10s}")
for name in ["dm", "ipw", "dr"]:
    v_base = policy_values[name]
    v_bad  = vals_bad_dm[name]
    print(f"{name.upper():8s}  {v_base:.6f}         {v_bad:.6f}      {v_bad - v_base:+.6f}")
print(f"  DR(bad DM)  = {vals_bad_dm['dr']:.6f}")
print(f"  IPS(good)   = {policy_values['ipw']:.6f}")
print(f"  V*          = {V_STAR}")


## Wizualizacja — Robustness DR


In [ ]:
scenarios = {
    "Baseline\n(dobry PS + DM)": {
        "DM": policy_values["dm"],
        "IPS": policy_values["ipw"],
        "DR": policy_values["dr"],
    },
    "Zły PS model\n(DM ratuje DR)": {
        "DM": vals_bad_ps["dm"],
        "IPS": vals_bad_ps["ipw"],
        "DR": vals_bad_ps["dr"],
    },
    "Zły reward model\n(IPS ratuje DR)": {
        "DM": vals_bad_dm["dm"],
        "IPS": vals_bad_dm["ipw"],
        "DR": vals_bad_dm["dr"],
    },
}

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
colors_est = {"DM": "steelblue", "IPS": "darkorange", "DR": "mediumpurple"}

for ax, (scenario, vals) in zip(axes, scenarios.items()):
    bars = ax.bar(vals.keys(), vals.values(),
                  color=[colors_est[e] for e in vals.keys()], alpha=0.85)
    ax.axhline(V_STAR, color="red", linestyle="--", linewidth=1.5, label=f"V* = {V_STAR}")
    ax.set_title(scenario, fontsize=10)
    ax.set_ylabel("Policy value V̂")
    ax.legend(fontsize=8)
    for bar, (name, val) in zip(bars, vals.items()):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0002,
                f"{val:.4f}", ha="center", va="bottom", fontsize=8)

plt.suptitle("Week 9: Doubly Robust — robustness na złe modele", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "dr_robustness.png", dpi=160, bbox_inches="tight")
plt.show()


## Tabela finalna — DM vs IPS vs SNIPS vs DR


In [ ]:
final_rows = []
for display, key in [("DM", "dm"), ("IPS", "ipw"), ("SNIPS", "snipw"), ("DR", "dr")]:
    v  = policy_values[key]
    ci = ci_results[key]
    final_rows.append({
        "Estimator":  display,
        "V̂":         v,
        "CI Lower":   ci["95.0% CI (lower)"],
        "CI Upper":   ci["95.0% CI (upper)"],
        "CI Width":   ci["95.0% CI (upper)"] - ci["95.0% CI (lower)"],
        "Bias vs V*": v - V_STAR,
    })

final_df = pd.DataFrame(final_rows)
print(f"Ground truth V* = {V_STAR}\n")
print(final_df.to_string(index=False, float_format="{:.6f}".format))
